## EJERCICIO 1

In [9]:
#En una terminal     
# nc -lk 9999

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
        .appName("Streaming WordCount") \
        .config("spark.streaming.stopGracefullyOnShutdown", "true") \
        .config("spark.sql.shuffle.partitions", 3) \
        .getOrCreate()

# Creamos un flujo de escucha sobre netcat en localhost:9999, mediante readStream
lineasDF = spark.readStream \
        .format("socket") \
        .option("host", "localhost") \
        .option("port", "9999") \
        .load()

# Leemos las líneas y las pasamos a palabras y realizamos la agrupación count (transformación)
palabrasDF = lineasDF.select(explode(split(lineasDF.value, ' ')).alias('palabra'))
cantidadDF = palabrasDF.groupBy("palabra").count()

# En Spark Streaming, la persistencia se realiza mediante writeStream
spark = SparkSession.builder 
wordCountQuery = cantidadDF.writeStream \
    .queryName("Caso1WordCount") \
    .format("console") \
    .outputMode("complete") \
    .start()

# dejamos Spark a la escucha
wordCountQuery.awaitTermination()

25/02/20 19:26:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/02/20 19:26:51 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
25/02/20 19:26:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-40086b50-b903-40b3-8657-7eee4e10a20f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/02/20 19:26:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


IllegalArgumentException: Cannot start query with name Caso1WordCount as a query with that name is already active in this SparkSession

## EJERCICIO 2

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession \
        .builder \
        .appName("Streaming de Ficheros") \
        .master("local[*]") \
        .config("spark.streaming.stopGracefullyOnShutdown", "true") \
        .config("spark.sql.shuffle.partitions", 3) \
        .config("spark.sql.streaming.schemaInference", "true") \
        .getOrCreate()

raw_df = spark.readStream \
        .format("json") \
        .option("path", "/home/iabd/Escritorio/IABD/Big-Data/Sistemas BD/UD 4/invoices") \
        .load()

raw_df.printSchema()

25/02/20 19:44:01 WARN Utils: Your hostname, iadb-04 resolves to a loopback address: 127.0.1.1; using 172.20.104.36 instead (on interface wlp44s0f0)
25/02/20 19:44:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/20 19:44:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- CESS: double (nullable = true)
 |-- CGST: double (nullable = true)
 |-- CashierID: string (nullable = true)
 |-- CreatedTime: long (nullable = true)
 |-- CustomerCardNo: string (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- DeliveryAddress: struct (nullable = true)
 |    |-- AddressLine: string (nullable = true)
 |    |-- City: string (nullable = true)
 |    |-- ContactNumber: string (nullable = true)
 |    |-- PinCode: string (nullable = true)
 |    |-- State: string (nullable = true)
 |-- DeliveryType: string (nullable = true)
 |-- InvoiceLineItems: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- ItemCode: string (nullable = true)
 |    |    |-- ItemDescription: string (nullable = true)
 |    |    |-- ItemPrice: double (nullable = true)
 |    |    |-- ItemQty: long (nullable = true)
 |    |    |-- TotalValue: double (nullable = true)
 |-- InvoiceNumber: string (nullable = true)
 |-- NumberOfItems: long (nullable = t

In [2]:
explode_df = raw_df.selectExpr("InvoiceNumber", "CreatedTime", "StoreID","PosID", "CustomerType",
                                 "PaymentMethod", "DeliveryType", "explode(InvoiceLineItems) as LineItem")
explode_df.printSchema()

root
 |-- InvoiceNumber: string (nullable = true)
 |-- CreatedTime: long (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- PosID: string (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- DeliveryType: string (nullable = true)
 |-- LineItem: struct (nullable = true)
 |    |-- ItemCode: string (nullable = true)
 |    |-- ItemDescription: string (nullable = true)
 |    |-- ItemPrice: double (nullable = true)
 |    |-- ItemQty: long (nullable = true)
 |    |-- TotalValue: double (nullable = true)



In [3]:
from pyspark.sql.functions import expr
limpio_df = explode_df \
    .withColumn("ItemCode", expr("LineItem.ItemCode")) \
    .withColumn("ItemDescription", expr("LineItem.ItemDescription")) \
    .withColumn("ItemPrice", expr("LineItem.ItemPrice")) \
    .withColumn("ItemQty", expr("LineItem.ItemQty")) \
    .withColumn("TotalValue", expr("LineItem.TotalValue")) \
    .drop("LineItem")
limpio_df.printSchema()

root
 |-- InvoiceNumber: string (nullable = true)
 |-- CreatedTime: long (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- PosID: string (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- DeliveryType: string (nullable = true)
 |-- ItemCode: string (nullable = true)
 |-- ItemDescription: string (nullable = true)
 |-- ItemPrice: double (nullable = true)
 |-- ItemQty: long (nullable = true)
 |-- TotalValue: double (nullable = true)



In [4]:
facturaWriterQuery = limpio_df.writeStream \
    .format("json") \
    .queryName("Facturas Writer") \
    .outputMode("append") \
    .option("path", "/home/iabd/Escritorio/IABD/Big-Data/Sistemas BD/UD 4/salida") \
    .option("checkpointLocation", "chk-point-dir-caso2-2") \
    .trigger(processingTime="1 minute") \
    .start()

25/02/20 19:44:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [5]:
raw_df = spark.readStream \
    .format("json") \
    .option("path", "/home/iabd/Escritorio/IABD/Big-Data/Sistemas BD/UD 4/invoices") \
    .option("maxFilesPerTrigger", 1) \
    .option("cleanSource", "delete") \
    .load()


In [6]:
facturaWriterQuery.explain() # muestra una explicación detalla del plan de ejecución
facturaWriterQuery.recentProgress # muestra una lista de los últimos progresos de la consulta
facturaWriterQuery.lastProgress # muestra el último progreso

No physical plan. Waiting for data.


{'id': '16a606fd-c11f-4e93-823f-024d89c8c19f',
 'runId': '3bbd5cdd-243b-4d0d-ab98-4548ffc7653c',
 'name': 'Facturas Writer',
 'timestamp': '2025-02-20T18:44:04.747Z',
 'batchId': 2,
 'numInputRows': 0,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.0,
 'durationMs': {'latestOffset': 6, 'triggerExecution': 67},
 'stateOperators': [],
 'sources': [{'description': 'FileStreamSource[file:/home/iabd/Escritorio/IABD/Big-Data/Sistemas BD/UD 4/invoices]',
   'startOffset': {'logOffset': 1},
   'endOffset': {'logOffset': 1},
   'latestOffset': None,
   'numInputRows': 0,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.0}],
 'sink': {'description': 'FileSink[/home/iabd/Escritorio/IABD/Big-Data/Sistemas BD/UD 4/salida]',
  'numOutputRows': -1}}

25/02/20 19:44:17 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## EJERCICIO 3

## EJERCICIO 4

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, max
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("BizumStreaming") \
    .getOrCreate()

# Definir el esquema del CSV
schema = StructType([
    StructField("Nombre", StringType(), True),
    StructField("Cantidad", IntegerType(), True),
    StructField("Concepto", StringType(), True)
])

# Leer los datos desde el stream (simulación de Bizum desde archivos en una carpeta)
df_stream = spark.readStream \
    .option("sep", ";") \
    .schema(schema) \
    .csv("/home/iabd/Escritorio/IABD/Big-Data/Sistemas BD/UD 4/bizums/recibidos")

# Normalizar nombres y obtener el máximo bizum recibido por persona
df_resultado = df_stream \
    .withColumn("Nombre", upper(col("Nombre"))) \
    .groupBy("Nombre") \
    .agg(max("Cantidad").alias("MaxBizum"))

# Mostrar resultados en la consola
query = df_resultado.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()


25/02/20 20:20:11 WARN Utils: Your hostname, iadb-04 resolves to a loopback address: 127.0.1.1; using 172.20.104.36 instead (on interface wlp44s0f0)
25/02/20 20:20:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/20 20:20:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/02/20 20:20:13 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-992a3e57-37ae-42c2-8c84-fc29431915f7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/02/20 20:20:13 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming Dat

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|   ANA|     300|
+------+--------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|   ANA|     300|
| RAMON|      90|
|  PEPE|      10|
+------+--------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|   ANA|     300|
| RAMON|      90|
| MARIA|      10|
|  PEPE|      10|
+------+--------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|   ANA|     300|
| RAMON|      90|
| MARIA|      10|
|  JUAN|       4|
|  PEPE|      10|
+------+--------+

-------------------------------------------
Batch: 4
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|   ANA|     300|
| RAMON|      90|
| JORGE|      20|
| MARIA|     300|
|  JUAN|       4|
|  PEPE|      10|
+------+--------+

-------------------------------------------
Batch: 5
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|   ANA|     300|
| RAMON|      90|
| JORGE|      20|
| MARIA|     300|
|  JUAN|       4|
|  PEPE|      10|
| ELENA|      23|
+------+--------+

-------------------------------------------
Batch: 6
-------------------------------------------
+------+--------+
|Nombre|MaxBizum|
+------+--------+
|ISABEL|      4

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/iabd/anaconda3/envs/IABD2/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=65>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/iabd/anaconda3/envs/IABD2/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/iabd/anaconda3/envs/IABD2/lib/python3.12/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/iabd/anaco

Py4JError: An error occurred while calling o51.awaitTermination

## EJERCICIO 5

In [37]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("Ventana fija IABD WordCount") \
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4") \
        .master("local[*]") \
        .config("spark.streaming.stopGracefullyOnShutdown", "true") \
        .config("spark.sql.shuffle.partitions", 3) \
        .getOrCreate()

dfLineas = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .option('includeTimestamp', 'true')\
    .load()

25/02/24 17:18:24 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [38]:
dfLineas.printSchema()

root
 |-- value: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [39]:
from pyspark.sql.functions import explode, split
dfPalabras = dfLineas.select(
    explode(split(dfLineas.value, ' ')).alias('palabra'),
    dfLineas.timestamp)


In [40]:
"""

Ventana fija

from pyspark.sql.functions import window
windowedCounts = dfPalabras.groupBy(
    window(dfPalabras.timestamp, "3 minutes"), dfPalabras.palabra
).count().orderBy('window')
"""
#Venta deslizante
from pyspark.sql.functions import window
windowedCounts = dfPalabras.groupBy(
    window(dfPalabras.timestamp, "3 minutes", "2 minute"), dfPalabras.palabra
).count().orderBy('window')

In [41]:
palabrasQuery = windowedCounts.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option('truncate', 'false')\
    .start()

25/02/24 17:18:24 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2f3221a6-4e15-4bcc-9639-12639b90d5fd. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/02/24 17:18:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+-------+-----+
|window|palabra|count|
+------+-------+-----+
+------+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-------+-----+
|window                                    |palabra|count|
+------------------------------------------+-------+-----+
|{2025-02-24 17:16:00, 2025-02-24 17:19:00}|Hola   |1    |
|{2025-02-24 17:16:00, 2025-02-24 17:19:00}|mundo  |1    |
|{2025-02-24 17:18:00, 2025-02-24 17:21:00}|Hola   |1    |
|{2025-02-24 17:18:00, 2025-02-24 17:21:00}|mundo  |1    |
+------------------------------------------+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-------+-----+
|window                                    |palabra|count|
+------------------------

25/02/24 17:21:01 WARN TextSocketMicroBatchStream: Stream closed by localhost:9999


## EJERCICIO 6